In [ ]:
import pandas as pd
import numpy as np


In [ ]:
# Read in RASMI material intensity data and MI from old IMAGE-Materials CP
mi_rasmi = pd.read_excel("MI_ranges_20230905.xlsx", index_col=[0, 1, 2, 3, 4], 
                         sheet_name=["concrete", "brick", "wood", "steel", "glass", "plastics", "aluminum", "copper"])

mi_image_mat = pd.read_csv("Building_materials_old.csv", index_col=[0, 1, 2])
# remove all data
mi_image_mat.loc[:, :] = np.nan

mi_image_mat_commercial = pd.read_csv("materials_commercial_old.csv", index_col=[0, 1, 2])


In [ ]:
def expand_image_mat_mi(mi_image_mat: pd.DataFrame,):
    # also get 2030 data in there
    # Get all unique index values for Region and HousingType
    regions = mi_image_mat.index.get_level_values(1).unique()
    housing_types = mi_image_mat.index.get_level_values(2).unique()

    # Create new index tuples for 2030
    new_index = pd.MultiIndex.from_product([[2030], regions, housing_types], names=mi_image_mat.index.names)

    # Add missing 2030 rows if not present
    mi_image_mat = mi_image_mat.reindex(mi_image_mat.index.union(new_index))

    # Now you can safely assign values for 2030
    mi_image_mat.loc[(2030, slice(None), slice(None)), :] = mi_image_mat.loc[(2020, slice(None), slice(None)), :].values
    return mi_image_mat

In [ ]:
# dictionary that maps RASMI R32 Regions to IMAGE R26 Regions

image_to_rasmi = {
    1: 'OECD_CAN',
    2: 'OECD_USA',
    3: 'LAM_MEX',
    4: 'LAM_LAM-L',
    5: 'LAM_BRA',
    6: 'LAM_LAM-L',
    7: ['MAF_MEA-H', 'MAF_MEA-M', 'MAF_NAF'],
    8: ['MAF_SSA-L', 'MAF_SSA-M'],
    9: ['MAF_SSA-L', 'MAF_SSA-M'],
    10: 'MAF_SAF',
    11: ['OECD_EFTA', 'OECD_EU12-H', 'OECD_EU12-M', 'OECD_EU15'],
    12: 'REF_EEU-FSU',
    13: 'OECD_TUR',
    14: ['REF_EEU-FSU', 'ASIA_TWN'],
    15: ['OECD_EEU', 'REF_CAS'],
    16: 'REF_RUS',
    17: ['MAF_MEA-H', 'MAF_MEA-M'],
    18: ['ASIA_IND', 'ASIA_OAS-CPA', 'ASIA_PAK'],
    19: 'OECD_KOR',
    20: 'ASIA_CHN',
    21: ['ASIA_OAS-L', 'ASIA_OAS-M'],
    22: 'ASIA_IDN',
    23: 'OECD_JPN',
    24: 'OECD_AUNZ',
    25: ['ASIA_OAS-L', 'ASIA_OAS-M'],
    26: ['MAF_SSA-L', 'MAF_SSA-M']
}

housing_type_image_to_rasmi = {
    1 : 'RS', #detached house to residential single family
    2 : 'RS', # semi detached house to residential single family
    3 : 'RM', # appartement to residential multi-family
    4 : 'RM', # high-rise to residential multi-family
}

housing_type_rasmi_to_image = {
    'RS' : [1, 2], # detached house and semi detached house to
    'RM' : [3, 4]  # appartement and high-rise to
}

housing_type_to_rasmi_building_structure = {
    1 : ['C', 'M', 'S', 'T'], # assumption that detached housing are average all structures
    2 : ['C', 'M', 'S', 'T'], # assumption that semi detached housing are average all structures
    3 : ['C', 'S'], # assumption that appartement are only made out of cement and steel structures
    4 : ['S'] # assumption that high-rise only steel structures
}

material_list_rasmi = ["steel", "concrete", "wood", "copper", "aluminum", "glass", "brick"]
mis_list_target = ["Steel", "Concrete", "Wood", "Copper", "Aluminium", "Glass", "Brick"]

# regions where total steel consumption is higher than what is estimated by image-materials sectors buildings & vehicles
steel_regions_adapt = [4, 8, 9, 22, 25, 26]